# ATM 407: the column at work

**This is the second of two notebooks.** The first, `01_meet_the_column.ipynb`, introduces each parameterization on its own: what radiation does to the column, what the boundary layer does, what convection does, each in isolation. This one puts them together and lets the column run. If you have not worked through the first notebook, start there -- the experiments below assume you know what the pieces are.

A single atmospheric column inside a global model is pushed by two things at once. Large-scale ascent supplied by the dynamical core lifts and destabilizes it; the physics parameterizations work to restore a balanced state. This notebook is about the interaction: you will diagnose the balance the column already holds, implement the dynamical forcing yourself, perturb the column, and interpret what it does with the diagnostics.

### Learning goals

By the end of the lab you should be able to:

1. connect hydrostatic balance to atmospheric mass on a sigma grid;
2. diagnose static stability using potential temperature and $N^2$;
3. distinguish resolved dynamical forcing from parameterized physics tendencies;
4. interpret longwave and shortwave heating profiles, TOA and surface fluxes, and cloud radiative effect;
5. manipulate temperature, humidity, condensate, and cloud-fraction profiles;
6. implement vertical advection and adiabatic temperature change in pressure coordinates; and
7. explain how forcing and convective-adjustment timescales control CAPE and rainfall.

The SCM has no horizontal pressure-gradient force, Coriolis acceleration, or internally resolved circulation. Here, large-scale vertical motion is supplied externally, exactly as an observationally forced SCM or a host dynamical core would supply it.

This model uses **mass-flux convection**, not Betts--Miller moist convective adjustment. Both express the same stabilizing principle: convection acts to remove convective instability. Betts--Miller directly relaxes temperature and humidity toward reference profiles; this scheme instead diagnoses a rising, entraining plume and chooses its mass flux to reduce CAPE over a specified timescale. The experiments therefore teach convective adjustment through a more process-based route.

**Lab rule:** write down what you expect before every experiment. A wrong prediction with a good physical explanation is worth more than a correct guess made after seeing the answer.

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/evanwellmeyer/GCM/blob/main/notebooks/02_experiments_atm407.ipynb)


> **Instructor note.** This notebook loads `scm/configs/atm407.toml` and starts from the accepted conservative mass-flux reference in `notebooks/data/atm407_equilibrium_20level.json`. Its 50-day means pass the equilibrium gates: TOA +0.93 W/m2, surface +0.41 W/m2, and surface-temperature drift 0.038 K. The atmosphere was equilibrated with a 5 m slab for speed; experiments use the depth stated in each section.

> Corrections in the column physics: cloud radiative effects follow the configuration switch; absorbers are distributed by atmospheric structure rather than level count; boundary-layer mixing conserves moist static energy and uses a diagnosed depth; condensation permits partial cloud below grid-mean saturation; and deep convection uses one conservative interface-flux transport for heat, vapor, downdrafts, and rain without an artificial moisture export or global energy repair.

> **Known limitations.** This remains a clear-sky, 20-level teaching model with simplified multiband radiation and parameterized convection. It now avoids the former saturated lower-tropospheric band, but it is not a replacement for an observed sounding or a comprehensive climate model. See `docs/column_open_problems.md`.


## Colab setup

Run this cell first. It installs the current model and verifies that Python can import it.

In [ ]:
#@title Install and connect the SCM { display-mode: "form" }
from pathlib import Path
import json
import subprocess
import sys

incolab = 'google.colab' in sys.modules
if incolab:
    root = Path('/content/GCM')
    if not root.exists():
        subprocess.run([
            'git', 'clone', '--depth', '1',
            'https://github.com/evanwellmeyer/GCM.git', str(root),
        ], check=True)
else:
    root = Path.cwd().resolve()
    while root != root.parent and not (root / 'pyproject.toml').exists():
        root = root.parent

if not (root / 'pyproject.toml').exists():
    raise FileNotFoundError('Open this notebook from inside the GCM repository.')
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

from importlib import metadata, util

try:
    metadata.version('gcm-scm')
    installed = util.find_spec('matplotlib') is not None
except metadata.PackageNotFoundError:
    installed = False

if not installed:
    subprocess.run([
        sys.executable, '-m', 'pip', 'install', '--quiet', '-e', f'{root}[plot]',
    ], check=True)
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

import scm
print('SCM ready')

In [ ]:
#@title Imports and visual style { display-mode: "form" }
from copy import deepcopy
import time

import matplotlib.pyplot as plt
from matplotlib import animation
from IPython.display import HTML, display
import numpy as np
import torch

from scm.column_model import initial_state, physics_step, run, update_derived
from scm.configuration import extract_param_overrides, load_run_config
from scm.ensemble import default_params
from scm.thermo import Lv, Rd, cp, g, geopotential, make_grid, relative_humidity
from scm.radiation_schemes.multiband import (
    compute_longwave_multiband, compute_shortwave_multiband,
)

torch.manual_seed(0)
device = torch.device('cpu')
plt.rcParams.update({
    'figure.facecolor': '#f7f3ea',
    'axes.facecolor': '#fffdf7',
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.titleweight': 'bold',
    'font.size': 11,
})
colors = {
    'sky': '#3a86ff', 'storm': '#264653', 'rain': '#00a896',
    'heat': '#ef476f', 'sun': '#ffb703', 'cloud': '#8d99ae',
}
print('device:', device)

## The column, back again

This is the same column you took apart in [`01_meet_the_column.ipynb`](https://colab.research.google.com/github/evanwellmeyer/GCM/blob/main/notebooks/01_meet_the_column.ipynb), with every scheme switched on at once and allowed to run forward in time. There the question was *what does each rule do*; here it is *what do they do to each other*.

We begin from a saved, nearly steady radiative-convective column rather than spending class time on a long spin-up. Radiation, surface fluxes, and convection are already active, so this is a moving balance--not a motionless atmosphere.

The reference has small energy imbalances, moderate CAPE, precipitation dominated by deep convection, and no active numerical limiters. It is an idealized laboratory atmosphere, not an observed climatological sounding. Think of it as the control member against which every forecast will be judged.

The saved equilibrium has 20 native levels, so the main lab keeps that scientifically consistent grid. A smoother-looking 40-level interpolation is not automatically a better atmosphere: remapping can alter stability and disturb the balance. You can investigate that distinction in the optional boss level.

The code cell below contains plumbing used throughout the lab. You do not need to memorize it. The governing equations that you are responsible for will be derived and implemented later in the lesson.


In [ ]:
#@title Lab helper functions { display-mode: "form" }
experiment = {
    'nlevels': 20,
    'dt': 900.0,
    'days': 3,
    'diagnostic_hours': 3,
    'radiation_steps': 8,
    'surface_temperature': 290.0,
    'surface_pressure': 100000.0,
    'solar_constant': 1360.0,
    'zenith_factor': 0.25,
    'ocean_depth': 50.0,
    'surface_albedo': 0.32,
    'wind_speed': 5.0,
}

referencemetadata = json.loads(
    (root / 'notebooks/data/atm407_equilibrium_20level.json').read_text()
)
print('reference configuration:', referencemetadata['configuration_label'])
print(f"reference surface temperature: {referencemetadata['surface_temperature_k']:.2f} K")
print(f"saved 20-level CAPE: {referencemetadata['cape_jkg']:.0f} J kg-1")
print(f"saved 20-level precipitation: {referencemetadata['precipitation_mmday']:.2f} mm day-1")
print(f"mass at or above 95% RH: {referencemetadata['rh95_mass_fraction']:.0%}")
print(f"mass-flux cap-active fraction: {referencemetadata['mass_flux_cap_fraction']:.0%}")

def makeparams(settings, updates=None):
    params = default_params(device=device)
    params.update(extract_param_overrides(load_run_config(root / 'scm/configs/atm407.toml')))
    params.update({
        'dt': settings['dt'],
        'ps0': settings['surface_pressure'],
        'ts_init': settings['surface_temperature'],
        'solar_constant': settings['solar_constant'],
        'zenith_factor': settings['zenith_factor'],
        'ocean_depth': settings['ocean_depth'],
        'albedo': settings['surface_albedo'],
        'wind_speed': settings['wind_speed'],
        'use_slab_ocean': True,
        'profile_diagnostics': True,
    })
    if updates is not None:
        params.update(updates)
    return params

def loadreference(nlevels=20, batch=1):
    reference = np.load(root / 'notebooks/data/atm407_equilibrium_20level.npz')
    settings = dict(experiment)
    settings['nlevels'] = nlevels
    grid = make_grid(nlevels, device=device)
    params = makeparams(settings)
    state = initial_state(batch, grid, params, device=device)
    sourcesigma = reference['sigma_full']
    targetsigma = grid['sigma_full'].cpu().numpy()

    for name in ['t', 'q', 'qc', 'cloud_fraction']:
        profile = np.interp(targetsigma, sourcesigma, reference[name])
        values = torch.as_tensor(profile, dtype=state[name].dtype, device=device)
        state[name] = values.unsqueeze(0).repeat(batch, 1)

    referencegrid = make_grid(len(sourcesigma), device=device)
    sourcedsigma = referencegrid['dsigma'].cpu().numpy()
    sourcewater = np.sum(reference['q'] * sourcedsigma)
    targetwater = torch.sum(state['q'][0] * grid['dsigma']).item()
    state['q'] = state['q'] * (sourcewater / targetwater)
    state['ts'].fill_(float(reference['ts']))
    state['ps'].fill_(float(reference['ps']))
    state['slab_ts_ref'] = state['ts'].clone()
    state['slab_energy'].zero_()
    return update_derived(state, grid)

def integrate(settings, updates=None, batch=1, state=None, lsforcing=None):
    grid = make_grid(settings['nlevels'], device=device)
    params = makeparams(settings, updates)
    if state is None:
        state = initial_state(batch, grid, params, device=device)
    stepsperday = round(86400 / settings['dt'])
    nsteps = round(settings['days'] * stepsperday)
    diagnosticsteps = max(1, round(settings['diagnostic_hours'] * 3600 / settings['dt']))
    start = time.perf_counter()
    state, history = run(
        state, grid, params, nsteps,
        rad_interval=settings['radiation_steps'],
        diag_interval=diagnosticsteps,
        ls_forcing=lsforcing,
    )
    elapsed = time.perf_counter() - start
    return grid, params, state, history, elapsed

def series(history, name, member=0, scale=1.0):
    values = [entry[name][member].detach().cpu().item() for entry in history]
    return np.array(values) * scale

def days(history, dt):
    return np.array([entry['step'] for entry in history]) * dt / 86400

# --- mission-independent helpers -------------------------------------------
# Any mission can call these, so the mission cells can be run in any order
# without tripping over a name that was defined inside an earlier mission.
_referencecache = {}
_baselinecache = {}

def referencecolumn(nlevels=20):
    """Grid, params and a fresh copy of the reference state (cached)."""
    if nlevels not in _referencecache:
        settings = dict(experiment)
        settings['nlevels'] = nlevels
        _referencecache[nlevels] = (
            make_grid(nlevels, device=device),
            makeparams(settings),
            loadreference(nlevels),
        )
    grid, params, state = _referencecache[nlevels]
    return grid, params, deepcopy(state)

def referencepressure(nlevels=20):
    """Full-level pressure of the reference column in hPa."""
    _, _, state = referencecolumn(nlevels)
    return state['p'][0].cpu().numpy() / 100

def baselinecolumn(nlevels=20):
    """Equilibrium CAPE (J/kg) and deep rain (mm/day) for the unforced column."""
    if nlevels not in _baselinecache:
        grid, params, state = referencecolumn(nlevels)
        _, diagnostics, _ = physics_step(state, grid, params)
        _baselinecache[nlevels] = (
            diagnostics['cape'][0].item(),
            diagnostics['precip_conv'][0].item() * 86400,
        )
    return _baselinecache[nlevels]


## Mission 1: weigh the atmosphere without a scale

Hydrostatic balance says $dp=-\rho g\,dz$. Integrating through one layer and dividing by area gives

$$m_{layer}/A=\frac{\Delta p}{g}.$$

That is why a pressure thickness is also a mass coordinate. This model uses $\sigma=p/p_s$, so its levels move with surface pressure while retaining a fixed fractional position in the column.

**Predict before plotting:** Are the layers equally spaced in pressure? Which portion of the atmosphere contains the most mass?

> **Your prediction:** Write one or two sentences here before running the cell.

Run the calculation, verify that $\sum\Delta p/g=p_s/g$, and explain why this identity is a useful conservation check for a dynamical core.


In [ ]:
grid = make_grid(experiment['nlevels'], device=device)
params = makeparams(experiment)
state = loadreference(experiment['nlevels'])
pressure = state['p'][0].cpu().numpy() / 100
deltap = state['dp'][0].cpu().numpy() / 100
levels = np.arange(experiment['nlevels'])
layercolors = plt.cm.Blues(0.25 + 0.65 * deltap / deltap.max())

fig, axes = plt.subplots(1, 2, figsize=(10, 5), sharey=True)
axes[0].plot(pressure, levels, marker='o', color=colors['storm'], linewidth=2.5)
axes[0].fill_betweenx(levels, 0, pressure, color=colors['sky'], alpha=0.12)
axes[0].set(xlabel='full-level pressure (hPa)', ylabel='model level', title='Pressure coordinates')
axes[1].barh(levels, deltap, color=layercolors, edgecolor='white')
axes[1].set(xlabel='layer pressure thickness (hPa)', title='Where the atmospheric mass lives')
axes[0].invert_yaxis()
for ax in axes:
    ax.grid(alpha=0.2)
axes[0].text(0.03, 0.94, 'TOP OF ATMOSPHERE', transform=axes[0].transAxes,
             color=colors['cloud'], fontweight='bold')
axes[0].text(0.72, 0.04, 'SURFACE', transform=axes[0].transAxes,
             color=colors['storm'], fontweight='bold')
fig.suptitle('A stack of air: thin aloft, heavy below', fontsize=16, color=colors['storm'])
fig.tight_layout()
plt.show()

massfromlayers = state['dp'].sum().item() / g
massfromsurface = state['ps'].item() / g
print(f'layer sum: {massfromlayers:.2f} kg m-2')
print(f'ps / g:    {massfromsurface:.2f} kg m-2')

## Mission 2: find the atmosphere's weak spots

A rising dry parcel cools by expansion, so temperature alone does not tell us whether it will return to its starting level. Potential temperature removes that adiabatic pressure effect:

$$\theta=T\left(\frac{p_0}{p}\right)^{R_d/c_p}.$$

The dry Brunt--Vaisala frequency measures the restoring force:

$$N^2=\frac{g}{\theta}\frac{\partial\theta}{\partial z}.$$

If $N^2>0$, a displaced parcel oscillates; if $N^2<0$, dry overturning is favored. Small $N^2$ marks a layer that is easy to disturb.

**Predict before plotting:** Where do you expect the strongest stability: boundary layer, free troposphere, or stratosphere?

> **Your prediction:** Record the layer and your physical reason.

Use the four panels to identify stable and unstable layers. A few negative values near the surface are not automatically a model failure: this is a discretized, turbulent boundary layer, while $N^2$ here is a dry parcel diagnostic. Treat them as a clue to discuss what the boundary-layer physics must continually mix. Then quantify the atmospheric mass at or above 95% RH and explain why this idealized column is not an observed tropical mean sounding.


In [ ]:
grid, params, state = referencecolumn()
pressure = referencepressure()
temperature = state['t'][0]
pressurepa = state['p'][0]
theta = temperature * (100000.0 / pressurepa) ** (Rd / cp)
rh = relative_humidity(state['q'], state['t'], state['p'])[0] * 100
height = geopotential(state['t'], state['q'], state['p'], grid)[0]
dthetadz = np.gradient(theta.cpu().numpy(), height.cpu().numpy())
n2 = g / theta.cpu().numpy() * dthetadz

fig, axes = plt.subplots(1, 4, figsize=(14, 5), sharey=True)
axes[0].plot(temperature.cpu(), pressure, color=colors['heat'], linewidth=2.5)
axes[0].set_xlabel('temperature (K)')
axes[1].plot(theta.cpu(), pressure, color=colors['sun'], linewidth=2.5)
axes[1].set_xlabel('potential temperature (K)')
axes[2].plot(rh.cpu(), pressure, color=colors['rain'], linewidth=2.5)
axes[2].plot(rh[rh >= 95].cpu(), pressure[rh.cpu().numpy() >= 95], 'o', color='tab:red')
axes[2].axvline(95, color='tab:red', linestyle='--', linewidth=0.8)
axes[2].set_xlabel('relative humidity (%)')
axes[3].plot(n2 * 1e4, pressure, color=colors['storm'], linewidth=2.5)
axes[3].fill_betweenx(pressure, 0, n2 * 1e4, where=n2 >= 0, color=colors['sky'], alpha=0.18, label='stable')
axes[3].fill_betweenx(pressure, 0, n2 * 1e4, where=n2 < 0, color=colors['heat'], alpha=0.25, label='dry unstable')
axes[3].axvline(0, color='black', linewidth=0.8)
axes[3].legend(fontsize=8, loc='upper left')
axes[3].set_xlabel(r'$N^2$ ($10^{-4}$ s$^{-2}$)')
axes[0].set_ylabel('pressure (hPa)')
axes[0].invert_yaxis()
for ax in axes:
    ax.grid(alpha=0.3)
fig.suptitle('Atmospheric health scan', fontsize=16, color=colors['storm'])
fig.tight_layout()
plt.show()
saturatedmass = state['dp'][0, rh >= 95].sum() / state['dp'][0].sum()
print(f'mass at or above 95% RH: {saturatedmass.item():.0%}')
print(f'height range: {height.min().item() / 1000:.1f} to {height.max().item() / 1000:.1f} km')
print(f'minimum dry N2: {n2.min():+.2e} s-2')
print('levels with dry N2 below zero (hPa):', np.round(pressure[n2 < 0], 1))

## Mission 3: become a tendency detective

A GCM usually separates its work into two pieces:

- the **dynamical core** transports mass, momentum, heat, and water;
- the **physics column** represents radiation, surface exchange, turbulence, convection, condensation, and clouds.

Radiation and surface exchange can change the column's total moist energy because they cross its boundaries. Mixing and convection mostly rearrange heat and water internally, so a column integral can hide large opposing tendencies at individual levels.

**Predict before plotting:** Which process should cool the upper atmosphere? Which should warm and moisten the lowest levels? Which should transport heat upward?

> **Your prediction:** Assign at least three processes to an expected sign and layer.

After running the cell, choose one surprising tendency and explain its sign in physical terms.


In [ ]:
#@title Reveal the physics tendency fingerprints { display-mode: "form" }
grid, params, state = referencecolumn()
pressure = referencepressure()
stepstate = deepcopy(state)
stepstate, diagnostics, radiationcache = physics_step(stepstate, grid, params)
processes = [
    ('radiation', 'radiation'), ('surface', 'surface'),
    ('boundary layer', 'boundary_layer'), ('shallow', 'shallow'),
    ('deep convection', 'deep'), ('condensation', 'condensation'),
    ('clouds', 'cloud'),
]
names = [label for label, key in processes]
temperaturemap = np.array([
    diagnostics[f'{key}_temperature_tendency'][0].cpu() * 86400
    for label, key in processes
])
moisturemap = np.array([
    diagnostics[f'{key}_moisture_tendency'][0].cpu() * 86400 * 1000
    for label, key in processes
])
boundaryvalues = np.array([
    diagnostics['rad_energy_tendency'][0].item(),
    diagnostics['surface_energy_tendency'][0].item(),
])

fig = plt.figure(figsize=(15, 6), constrained_layout=True)
layout = fig.add_gridspec(1, 3, width_ratios=[0.8, 1.5, 1.5])
axes = [fig.add_subplot(layout[0, index]) for index in range(3)]

barcolors = [colors['heat'] if value > 0 else colors['sky'] for value in boundaryvalues]
axes[0].bar(['radiative\ncooling', 'surface\nheating'], boundaryvalues, color=barcolors, width=0.65)
axes[0].axhline(0, color=colors['storm'], linewidth=1)
axes[0].set(title='Who pays the energy bill?', ylabel='column tendency (W m$^{-2}$)')
axes[0].grid(axis='y', alpha=0.2)

for ax, values, title, label in [
    (axes[1], temperaturemap, 'Temperature fingerprints', 'K day$^{-1}$'),
    (axes[2], moisturemap, 'Moisture fingerprints', 'g kg$^{-1}$ day$^{-1}$'),
]:
    limit = max(np.nanpercentile(np.abs(values), 98), 1e-6)
    image = ax.imshow(values.T, origin='upper', aspect='auto', cmap='RdBu_r', vmin=-limit, vmax=limit)
    ticklevels = np.linspace(0, len(pressure) - 1, 6).astype(int)
    ax.set_xticks(range(len(names)), names, rotation=40, ha='right')
    ax.set_yticks(ticklevels, [f'{pressure[index]:.0f}' for index in ticklevels])
    ax.set(title=title, ylabel='pressure (hPa)')
    fig.colorbar(image, ax=ax, shrink=0.72, label=label)

fig.suptitle('Physics fingerprint scan', fontsize=17, color=colors['storm'])
plt.show()
print(f"instantaneous TOA net flux: {diagnostics['toa_net'][0].item():+.2f} W m-2")
print(f"instantaneous surface flux: {diagnostics['surface_total_flux'][0].item():+.2f} W m-2")
print(f"50-day mean TOA net flux: {referencemetadata['toa_net_wm2']:+.2f} W m-2")
print(f"50-day mean surface flux: {referencemetadata['surface_total_flux_wm2']:+.2f} W m-2")
print(f"column residual: {diagnostics['column_energy_residual'][0].item():+.2f} W m-2")

## Mission 4: radiative accounting inside the column

Radiation does more than add a number at the top of the atmosphere. A layer warms when more radiative energy enters than leaves and cools when the flux diverges. In pressure coordinates, the radiative temperature tendency has the form

$$\left.\frac{\partial T}{\partial t}\right|_{\rm rad}
=-\frac{g}{c_p}\frac{\partial F_{\rm net}}{\partial p}.$$

The sign depends on the flux convention, but the physical rule is simple: **flux convergence warms; flux divergence cools**.

Shortwave radiation originates with the Sun and is absorbed or reflected. Longwave radiation is emitted by the surface and atmosphere. Their vertical profiles need not cancel locally even when the top-of-atmosphere budget is nearly closed.

**Predict before running:**

1. Will shortwave radiation warm or cool atmospheric layers?
2. Where should longwave cooling be strongest?
3. If CO$_2$ increases while the atmospheric state is held fixed, should outgoing longwave radiation initially rise or fall?

Use the sliders to perturb radiation instantaneously. The sounding, clouds, and surface temperature are held fixed, so this is a **radiative adjustment**, not a new climate equilibrium. The surface ledger includes radiation only; sensible and latent heat fluxes are needed to evaluate the complete surface energy balance.


In [ ]:
co2ppm = 400 #@param {type:"slider", min:200, max:800, step:50}
sunlightpercent = 100 #@param {type:"slider", min:90, max:110, step:1}

radiationparams = makeparams(experiment)
radiationparams['co2'] = float(co2ppm)
radiationparams['solar_constant'] = experiment['solar_constant'] * sunlightpercent / 100

In [ ]:
#@title Reveal the radiative accounting { display-mode: "form" }
grid, params, stepstate = referencecolumn()
stepstate, diagnostics, radiationcache = physics_step(stepstate, grid, params)
pressure = referencepressure()
lwheating, lwdown, olr = compute_longwave_multiband(stepstate, grid, radiationparams)
swheating, swsurface, asr, swreflected, toainsolation = compute_shortwave_multiband(
    stepstate, grid, radiationparams
)
netheating = lwheating + swheating
lwupsurface = diagnostics['lw_up_sfc'][0].item()
toanet = (asr - olr)[0].item()
surfacenet = swsurface[0].item() + lwdown[0].item() - lwupsurface

# Heating is flux divergence: dT/dt = -(g/cp) dF/dp. Running that backwards turns
# the heating profile into the net flux crossing every level in the column. The
# sweep starts from the known flux at the top of the atmosphere, so the value it
# lands on at the surface is a genuine check, not an assumption.
layerdp = stepstate['dp'][0].cpu().numpy()
halfpressure = (grid['sigma_half'].cpu().numpy() * stepstate['ps'][0].item()) / 100
lwnetflux = olr[0].item() + np.concatenate(
    [[0.0], np.cumsum(cp / g * lwheating[0].cpu().numpy() * layerdp)])
swnetflux = -asr[0].item() + np.concatenate(
    [[0.0], np.cumsum(cp / g * swheating[0].cpu().numpy() * layerdp)])
totalnetflux = lwnetflux + swnetflux
atmosphericabsorption = (asr[0].item() - swsurface[0].item())

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

lwprofile = lwheating[0].cpu().numpy() * 86400
swprofile = swheating[0].cpu().numpy() * 86400
netprofile = netheating[0].cpu().numpy() * 86400
axes[0, 0].plot(lwprofile, pressure, color=colors['heat'], linewidth=2.5, label='longwave')
axes[0, 0].plot(swprofile, pressure, color=colors['sun'], linewidth=2.5, label='shortwave')
axes[0, 0].plot(netprofile, pressure, color=colors['storm'], linewidth=3, label='net')
axes[0, 0].fill_betweenx(pressure, 0, netprofile, where=netprofile >= 0,
                         color=colors['sun'], alpha=0.18)
axes[0, 0].fill_betweenx(pressure, 0, netprofile, where=netprofile < 0,
                         color=colors['sky'], alpha=0.18)
axes[0, 0].axvline(0, color='black', linewidth=0.8)
axes[0, 0].invert_yaxis()
axes[0, 0].set(xlabel='radiative tendency (K day$^{-1}$)', ylabel='pressure (hPa)',
               title='Where radiation heats and cools')
axes[0, 0].legend(frameon=False)

axes[0, 1].plot(lwnetflux, halfpressure, color=colors['heat'], linewidth=2.5,
                label='net longwave')
axes[0, 1].plot(swnetflux, halfpressure, color=colors['sun'], linewidth=2.5,
                label='net shortwave')
axes[0, 1].plot(totalnetflux, halfpressure, color=colors['storm'], linewidth=3,
                label='total')
axes[0, 1].axvline(0, color='black', linewidth=0.8)
axes[0, 1].scatter([totalnetflux[0]], [halfpressure[0]], s=90, zorder=4,
                   facecolors='none', edgecolors=colors['rain'], linewidths=2.2)
axes[0, 1].scatter([totalnetflux[-1]], [halfpressure[-1]], s=90, zorder=4,
                   facecolors='none', edgecolors=colors['rain'], linewidths=2.2)
axes[0, 1].annotate('top of atmosphere', xy=(totalnetflux[0], halfpressure[0]),
                    xytext=(0.06, 0.10), textcoords='axes fraction', fontsize=9)
axes[0, 1].annotate('surface', xy=(totalnetflux[-1], halfpressure[-1]),
                    xytext=(0.06, 0.90), textcoords='axes fraction', fontsize=9)
axes[0, 1].invert_yaxis()
axes[0, 1].set(xlabel='net upward flux (W m$^{-2}$)', ylabel='pressure (hPa)',
               title='Radiation carried through every level')
axes[0, 1].legend(frameon=False, fontsize=9)

toavalues = [asr[0].item(), -olr[0].item(), toanet]
toacolors = [colors['sun'], colors['heat'], colors['rain'] if abs(toanet) < 1 else colors['cloud']]
toabars = axes[1, 0].bar(range(3), toavalues, color=toacolors, width=0.65)
axes[1, 0].set_xticks(range(3), ['absorbed solar', 'outgoing LW', 'net'], rotation=15)
axes[1, 0].bar_label(toabars, fmt='%.1f', padding=3, fontsize=9)
axes[1, 0].axhline(0, color='black', linewidth=0.8)
axes[1, 0].set(ylabel='downward-positive flux (W m$^{-2}$)', title='Top-of-atmosphere ledger')

surfacevalues = [swsurface[0].item(), lwdown[0].item(), -lwupsurface, surfacenet]
surfacebars = axes[1, 1].bar(
    range(4), surfacevalues,
    color=[colors['sun'], colors['heat'], colors['sky'], colors['storm']],
)
axes[1, 1].set_xticks(range(4), ['solar', 'LW down', 'LW up', 'net'], rotation=15)
axes[1, 1].bar_label(surfacebars, fmt='%.1f', padding=3, fontsize=9)
axes[1, 1].axhline(0, color='black', linewidth=0.8)
axes[1, 1].set(ylabel='downward-positive flux (W m$^{-2}$)', title='Surface radiative ledger')
for ax in axes.ravel():
    ax.grid(alpha=0.2)
fig.suptitle(f'Radiative accounting: {co2ppm} ppm CO$_2$, {sunlightpercent}% sunlight',
             fontsize=16, color=colors['storm'])
fig.tight_layout()
plt.show()

rating = 'BALANCED' if abs(toanet) < 1 else 'SURPLUS' if toanet > 0 else 'DEFICIT'
print(f'TOP OF ATMOSPHERE   absorbed solar {asr[0].item():7.2f}   outgoing LW {olr[0].item():7.2f}'
      f'   net {toanet:+.2f} W m-2')
print(f'ATMOSPHERE          absorbs {atmosphericabsorption:7.2f} W m-2 of sunlight directly')
print(f'SURFACE             solar {swsurface[0].item():7.2f}   LW down {lwdown[0].item():7.2f}'
      f'   LW up {lwupsurface:7.2f}   net {surfacenet:+.2f} W m-2')
print()
print(f'flux check: the sweep down from the top lands on {lwnetflux[-1]:.2f} W m-2 of net upward')
print(f'            longwave at the surface; the surface calculation gives '
      f'{lwupsurface - lwdown[0].item():.2f}')
print('surface note: sensible and latent heat fluxes are not part of this radiative ledger')
print(f'radiative-balance rating: {rating}')


### Radiative detective tasks

1. Identify the pressure level with the strongest longwave cooling and the level with the strongest shortwave heating. Relate each to temperature, water vapor, clouds, or ozone.
2. Double CO$_2$ from 400 to 800 ppm while holding sunlight fixed. Record the changes in outgoing longwave radiation, TOA net radiation, and the vertical heating profile.
3. Return to 400 ppm, change sunlight alone, and explain why its vertical fingerprint differs from the CO$_2$ perturbation.
4. Find **two different slider combinations** with `BALANCED` TOA ratings. Do their atmospheric heating profiles match? Explain why TOA closure is insufficient to describe the internal column response.
5. Predict how the surface temperature would eventually respond to a positive TOA imbalance. Why can this instantaneous calculation not determine the final temperature change?
6. Connect the radiative profile to Mission 3: which processes compensate radiative cooling in the lower troposphere, and does that compensation occur at exactly the same levels?

### Sounding and cloud-radiation laboratory

This focused experiment addresses a second question: how do the atmospheric sounding and clouds change the radiation tendencies? You may alter the supplied temperature and relative-humidity profiles below. The cloud controls prescribe a layer between 400 and 700 hPa and divide its condensate into liquid and ice optical paths. The model itself carries one total-condensate variable, `qc`, and partitions it by temperature; separate prognostic `qi` is not yet part of this SCM.

Cloud radiative effect (CRE) is an instantaneous diagnostic at the same atmospheric state: **all-sky minus clear-sky**. For the TOA net flux, positive CRE warms the column and negative CRE cools it. The radiation code uses random overlap, so the total cloud cover of layer fractions $F_k$ is $1-\prod_k(1-F_k)$. Maximum overlap would instead give $\max(F_k)$; comparing those values shows why overlap matters, although maximum-overlap radiative transfer is not implemented here.

To use your own sounding, replace `temperatureprofile` and `rhprofile` with 20 values ordered from the top of the atmosphere to the surface. A CSV can be loaded with `np.loadtxt`; interpolate its pressure, temperature, and RH columns to `pressure` before assigning them. Keep temperature positive and RH between 0 and 1.


In [ ]:
temperatureoffset = 0.0 #@param {type:"slider", min:-5, max:5, step:0.5}
rhscale = 1.0 #@param {type:"slider", min:0.5, max:1.2, step:0.05}
cloudfraction = 0.5 #@param {type:"slider", min:0, max:1, step:0.1}
liquidpath = 0.05 #@param {type:"slider", min:0, max:0.3, step:0.01}
icepath = 0.02 #@param {type:"slider", min:0, max:0.3, step:0.01}


In [ ]:
#@title Compare all-sky and clear-sky radiation { display-mode: "form"}
grid, cloudparams, cloudstate = referencecolumn()
pressure = referencepressure()
temperatureprofile = cloudstate['t'][0].clone() + float(temperatureoffset)
rhprofile = relative_humidity(cloudstate['q'], cloudstate['t'], cloudstate['p'])[0]
rhprofile = (rhprofile * float(rhscale)).clamp(min=0.0, max=1.0)
cloudstate['t'][0] = temperatureprofile
from scm.thermo import saturation_specific_humidity
cloudstate['q'][0] = rhprofile * saturation_specific_humidity(
    cloudstate['t'][0], cloudstate['p'][0])
cloudstate = update_derived(cloudstate, grid)

deck = (cloudstate['p'][0] >= 40000) & (cloudstate['p'][0] <= 70000)
fractions = torch.zeros_like(cloudstate['q'])
fractions[0, deck] = float(cloudfraction)
count = max(int(deck.sum().item()), 1)
cloudstate['cloud_fraction'] = fractions
cloudstate['cloud_sw_tau_layer'] = fractions * (
    80.0 * float(liquidpath) + 40.0 * float(icepath)) / count
cloudstate['cloud_lw_tau_layer'] = fractions * (
    12.0 * float(liquidpath) + 6.0 * float(icepath)) / count
cloudparams['cloud_radiative_effects_enabled'] = True
cloudparams['cloud_optics_scheme'] = 'microphysics'
cloudparams['cloud_fractional_gridmean_optics'] = True

alllw, alldown, allolr = compute_longwave_multiband(cloudstate, grid, cloudparams)
allsw, allsurface, allasr, _, _ = compute_shortwave_multiband(cloudstate, grid, cloudparams)
clearlw, cleardown, clearolr = compute_longwave_multiband(
    cloudstate, grid, cloudparams, force_clear_sky=True)
clearsw, clearsurface, clearasr, _, _ = compute_shortwave_multiband(
    cloudstate, grid, cloudparams, force_clear_sky=True)

alltoa = (allasr - allolr)[0].item()
cleartoa = (clearasr - clearolr)[0].item()
creprofile = (alllw + allsw - clearlw - clearsw)[0].cpu().numpy() * 86400
fig, axes = plt.subplots(1, 3, figsize=(14, 5), sharey=True)
axes[0].plot(temperatureprofile.cpu(), pressure, color=colors['heat'], linewidth=2.5)
axes[0].set(xlabel='temperature (K)', ylabel='pressure (hPa)', title='Edited sounding')
axes[1].plot(rhprofile.cpu() * 100, pressure, color=colors['sky'], linewidth=2.5)
axes[1].plot(fractions[0].cpu() * 100, pressure, color=colors['cloud'], linewidth=2.5, label='cloud fraction')
axes[1].set(xlabel='fraction (%)', title='Humidity and cloud')
axes[1].legend(frameon=False)
axes[2].plot(creprofile, pressure, color=colors['storm'], linewidth=2.5)
axes[2].axvline(0, color='black', linewidth=0.8)
axes[2].set(xlabel='all-sky minus clear-sky (K day$^{-1}$)', title='Cloud heating effect')
for ax in axes:
    ax.invert_yaxis()
    ax.grid(alpha=0.2)
plt.show()
randomcover = 1.0 - torch.prod(1.0 - fractions[0, deck]).item()
maximumcover = fractions[0, deck].max().item() if deck.any() else 0.0
print(f'TOA all-sky: {alltoa:+.2f} W m-2')
print(f'TOA clear-sky: {cleartoa:+.2f} W m-2')
allsurfacenet = (allsurface + alldown)[0].item()
clearsurfacenet = (clearsurface + cleardown)[0].item()
print(f'TOA cloud radiative effect: {alltoa - cleartoa:+.2f} W m-2')
print(f'surface cloud radiative effect: {allsurfacenet - clearsurfacenet:+.2f} W m-2')
print(f'random-overlap cover: {randomcover:.1%}; maximum-overlap cover: {maximumcover:.1%}')


#### Sounding and cloud tasks

1. Increase RH alone. Explain changes in LW heating, OLR, and TOA balance.
2. Add liquid cloud, then ice cloud, holding cloud fraction fixed. Compare the SW, LW, and net CRE.
3. Switch cloud fraction to zero. Verify that all-sky and clear-sky results match.
4. Change cloud fraction while holding condensate paths fixed. Explain why cloud amount and condensate amount are distinct controls.
5. Compare random- and maximum-overlap cloud cover. Which assumption exposes more clear sky when several layers contain cloud?
6. Edit one part of the temperature or RH profile, state your hypothesis first, and use the heating profile plus TOA and surface fluxes to test it.


### Radiation alone: the shape radiation would build

You have seen where radiation heats and cools, but not yet what radiation would *do* to the column if nothing stopped it. Now switch the other machinery off.

In the next experiment deep convection, shallow convection, and the dry adjustment are all disabled. Radiation, surface exchange, and boundary-layer mixing keep running. The surface temperature is held fixed, so the question is purely about the **shape of the atmosphere above it**: given this surface, what temperature profile does radiation build on its own?

Longwave cooling is strongest in the moist lower troposphere and weakens aloft, so radiation removes heat from the layers just above the surface faster than from the layers higher up. Nothing carries heat upward to replace it. Watch what that does to the lapse rate.

> **Predict before running:** Will the free troposphere warm or cool? Will the profile become more stable or less stable than the reference? Sketch the temperature curve you expect after 40 days next to the reference curve.

This is the classic radiative-equilibrium calculation of Manabe and Strickler (1964), and the answer it gave is the reason every atmospheric model has a convection scheme at all.


In [ ]:
radiationdays = 40 #@param {type:"slider", min:10, max:80, step:10}

print(f'running radiation-only physics for {radiationdays} days')
print('disabled: deep convection, shallow convection, dry adjustment')
print('active:   radiation, surface fluxes, boundary-layer mixing, condensation')


In [ ]:
#@title Run the radiation-only column { display-mode: "form" }
radiationsettings = dict(experiment)
radiationsettings['days'] = radiationdays
radiationsettings['diagnostic_hours'] = 24
radiationonlyupdates = {
    'convection_scheme': 'none',
    'shallow_convection_scheme': 'none',
    'dry_adjustment_enabled': False,
    'use_slab_ocean': False,          # hold the surface, ask only about the air above it
}
radiationgrid = make_grid(experiment['nlevels'], device=device)
radiationstate = loadreference(experiment['nlevels'])
referenceprofile = radiationstate['t'][0].cpu().numpy().copy()
_, _, radiationstate, radiationhistory, radiationelapsed = integrate(
    radiationsettings, updates=radiationonlyupdates, state=radiationstate
)
radiationprofile = radiationstate['t'][0].cpu().numpy()
pressure = referencepressure()

def lapserate(profile, pressureprofile):
    """environmental lapse rate between adjacent levels, in K per km."""
    lapse = np.zeros(len(profile) - 1)
    for level in range(len(profile) - 1):
        meantemperature = 0.5 * (profile[level] + profile[level + 1])
        thickness = (Rd * meantemperature / g
                     * np.log(pressureprofile[level + 1] / pressureprofile[level]))
        lapse[level] = -(profile[level] - profile[level + 1]) / thickness * 1000
    return lapse

referencelapse = lapserate(referenceprofile, pressure)
radiationlapse = lapserate(radiationprofile, pressure)
midpressure = 0.5 * (pressure[:-1] + pressure[1:])
dryadiabat = g / cp * 1000

fig, axes = plt.subplots(1, 3, figsize=(15, 5.5), sharey=True)
axes[0].plot(referenceprofile, pressure, color=colors['storm'], linewidth=3,
             label='full model (reference)')
axes[0].plot(radiationprofile, pressure, color=colors['heat'], linewidth=3,
             linestyle='--', label=f'radiation only, day {radiationdays}')
axes[0].fill_betweenx(pressure, referenceprofile, radiationprofile,
                      color=colors['sky'], alpha=0.15)
axes[0].set(xlabel='temperature (K)', ylabel='pressure (hPa)',
            title='Radiation alone cools the free troposphere')
axes[0].legend(frameon=False, fontsize=9)
axes[0].invert_yaxis()

axes[1].plot(referencelapse, midpressure, color=colors['storm'], linewidth=3,
             label='full model')
axes[1].plot(radiationlapse, midpressure, color=colors['heat'], linewidth=3,
             linestyle='--', label='radiation only')
axes[1].axvline(dryadiabat, color=colors['rain'], linewidth=2,
                label=f'dry adiabat ({dryadiabat:.1f} K km$^{{-1}}$)')
axes[1].fill_betweenx(midpressure, dryadiabat, np.maximum(radiationlapse, dryadiabat),
                      color=colors['heat'], alpha=0.20)
axes[1].set(xlabel='lapse rate (K km$^{-1}$)',
            title='Anything right of the red line\noverturns immediately')
axes[1].legend(frameon=False, fontsize=8)

warming = radiationprofile - referenceprofile
axes[2].plot(warming, pressure, color=colors['sky'], linewidth=3)
axes[2].axvline(0, color='black', linewidth=0.9)
axes[2].fill_betweenx(pressure, 0, warming, where=warming < 0,
                      color=colors['sky'], alpha=0.20)
axes[2].set(xlabel='temperature change (K)',
            title='How far the column drifted')
for ax in axes:
    ax.grid(alpha=0.25)
fig.suptitle('What radiation builds when nothing carries heat upward',
             fontsize=16, color=colors['storm'])
fig.tight_layout()
plt.show()

unstablelevels = int(np.sum(radiationlapse > dryadiabat))
print(f'steepest lapse rate, full model:      {referencelapse.max():6.1f} K km-1')
print(f'steepest lapse rate, radiation only:  {radiationlapse.max():6.1f} K km-1'
      f'   ({radiationlapse.max() / dryadiabat:.1f} times the dry adiabat)')
print(f'levels left unstable to dry overturning: {unstablelevels} of {len(radiationlapse)}')
print(f'largest cooling: {warming.min():.1f} K at {pressure[int(np.argmin(warming))]:.0f} hPa')
print(f'runtime: {radiationelapsed:.1f} s')


#### Radiation-alone tasks

1. Radiation cools the free troposphere strongly but the lowest layers barely change. Explain why, using the longwave heating profile you plotted earlier and the fact that the surface temperature was held fixed.
2. Compare the steepest lapse rate against the dry adiabat. A parcel displaced upward in that layer keeps accelerating. Explain, in one sentence, why an atmosphere cannot actually sit in this state.
3. This profile is a genuine equilibrium of the equations we left switched on, yet no such atmosphere is ever observed. What does that tell you about which processes set the observed lapse rate?
4. Increase the run length. Does the profile keep steepening without limit, or does it approach something? What sets the limit?
5. Now connect it back: in the full model, which tendencies from Mission 3 are removing this instability, and roughly how large must they be to hold the lapse rate near its reference value?
6. The reference column has a lapse rate well below the dry adiabat and is therefore stable to dry overturning, yet the model still produces deep convective rain. Explain what makes a *moist* parcel unstable when a dry one is not.


## Mission 5: build the dynamical forcing

Now you will write the bridge between dynamics and physics. In pressure coordinates, $\omega=Dp/Dt$ is negative for ascent. Ignoring horizontal advection,

$$\frac{\partial T}{\partial t}=\underbrace{-\omega\frac{\partial T}{\partial p}}_{\text{vertical advection}}+\underbrace{\frac{R_d}{c_p}\frac{T\omega}{p}}_{\text{adiabatic expansion}},$$

$$\frac{\partial q}{\partial t}=-\omega\frac{\partial q}{\partial p}.$$

The first temperature term moves the environmental profile through the column. The second is compressional heating or expansional cooling. Their sum determines the local temperature response.

We prescribe $\omega(p)$ to be zero at the top and surface and strongest near $\sigma=0.5$. It acts for one day; the column then adjusts freely for two days.

**Predict before coding:** For ascent, predict the signs of vertical temperature advection, adiabatic expansion, total temperature tendency, and moisture tendency. Will CAPE and deep rain increase or decrease?

> **Your prediction:** Make a sign table and predict the CAPE/rain response.

After the static diagnosis, press play on the animated storm dashboard. The blue shading marks the day when large-scale ascent is switched on.

### Storm control panel

The next cell exposes three bounded Colab sliders:

- **ascent strength** controls how rapidly dynamics pushes air upward;
- **forcing duration** controls how long that push persists;
- **convective timescale** controls how quickly the parameterization consumes CAPE.

Change one slider at a time, rerun the control panel, and then rerun the two collapsed result cells below it. This is a controlled sensitivity experiment, not random tuning.


In [ ]:
# First implement the pressure derivatives used in the equations above.
def pressuregradient(field, pressure):
    gradient = torch.zeros_like(field)
    gradient[:, 1:-1] = (field[:, 2:] - field[:, :-2]) / (pressure[:, 2:] - pressure[:, :-2])
    gradient[:, 0] = (field[:, 1] - field[:, 0]) / (pressure[:, 1] - pressure[:, 0])
    gradient[:, -1] = (field[:, -1] - field[:, -2]) / (pressure[:, -1] - pressure[:, -2])
    return gradient

# This function recomputes the forcing from the evolving model state.
def ascentforcing(grid, peakomega, durationdays=1.0):
    peakomega = torch.as_tensor(peakomega, dtype=torch.float32, device=device).reshape(-1, 1)
    durationdays = torch.as_tensor(durationdays, dtype=torch.float32, device=device).reshape(-1, 1)
    sigma = grid['sigma_full'].to(device=device, dtype=torch.float32).reshape(1, -1)
    shape = torch.sin(torch.pi * sigma).clamp(min=0.0)

    def forcing(step, state):
        active = step * experiment['dt'] < durationdays * 86400
        if not torch.any(active):
            return None
        pressure = state['p']
        omega = -peakomega.to(pressure.dtype) * shape.to(pressure.dtype)
        omega = omega * active.to(pressure.dtype)
        dtdp = pressuregradient(state['t'], pressure)
        dqdp = pressuregradient(state['q'], pressure)

        verticaladvection = -omega * dtdp
        adiabaticexpansion = (Rd / cp) * state['t'] * omega / pressure
        temperaturetendency = verticaladvection + adiabaticexpansion
        moisturetendency = -omega * dqdp
        return {
            'dt': temperaturetendency,
            'dq': moisturetendency,
            'verticaladvection': verticaladvection,
            'adiabaticexpansion': adiabaticexpansion,
        }

    return forcing

In [ ]:
ascentstrength = 1.80 #@param {type:"slider", min:0.36, max:3.60, step:0.18}
forcingdays = 1.00 #@param {type:"slider", min:0.25, max:2.00, step:0.25}
tauhours = 6.0 #@param {type:"slider", min:0.5, max:12, step:0.5}

grid = make_grid(experiment['nlevels'], device=device)
peakomega = ascentstrength / 36
experimentupdates = {
    'tau_cape': torch.tensor([tauhours * 3600.0], device=device),
}
forcing = ascentforcing(grid, [peakomega], durationdays=forcingdays)
print(f'ascent: {ascentstrength:.2f} hPa hour-1')
print(f'forcing duration: {forcingdays:.2f} days')
print(f'convective timescale: {tauhours:.0f} hours')

In [ ]:
#@title Run and reveal the ascent experiment { display-mode: "form" }
pressure = referencepressure()
previewstate = loadreference(experiment['nlevels'])
preview = forcing(0, previewstate)
experimentparams = makeparams(experiment, experimentupdates)
baselinestate, baselinediagnostics, baselinecache = physics_step(
    deepcopy(previewstate), grid, experimentparams
)
baselinecape = baselinediagnostics['cape'][0].item()
baselinerain = baselinediagnostics['precip_conv'][0].item() * 86400
omega = -peakomega * np.sin(np.pi * grid['sigma_full'].cpu().numpy())

fig, axes = plt.subplots(1, 4, figsize=(15, 5), sharey=True)
axes[0].plot(omega * 36, pressure, color=colors['sky'], linewidth=3)
axes[0].fill_betweenx(pressure, omega * 36, 0, color=colors['sky'], alpha=0.18)
axes[0].set(xlabel=r'$\omega$ (hPa hour$^{-1}$)', ylabel='pressure (hPa)')
axes[1].plot(preview['verticaladvection'][0].cpu() * 86400, pressure, color=colors['heat'], linewidth=2.5, label='vertical advection')
axes[1].plot(preview['adiabaticexpansion'][0].cpu() * 86400, pressure, color=colors['sky'], linewidth=2.5, label='adiabatic expansion')
axes[1].set_xlabel('temperature terms (K day-1)')
axes[1].legend(fontsize=8)
axes[2].plot(preview['dt'][0].cpu() * 86400, pressure, color=colors['storm'], linewidth=3)
axes[2].set_xlabel('total temperature tendency (K day-1)')
axes[3].plot(preview['dq'][0].cpu() * 86400 * 1000, pressure, color=colors['rain'], linewidth=3)
axes[3].set_xlabel('moisture tendency (g kg-1 day-1)')
for ax in axes:
    ax.axvline(0, color='black', linewidth=0.8)
    ax.grid(alpha=0.3)
axes[0].invert_yaxis()
fig.suptitle('Anatomy of imposed ascent', fontsize=17, color=colors['storm'])
fig.tight_layout()
plt.show()

forcedstate = loadreference(experiment['nlevels'])
grid, params, forcedstate, forcedhistory, forcedelapsed = integrate(
    experiment, updates=experimentupdates, state=forcedstate, lsforcing=forcing
)
timeaxis = days(forcedhistory, experiment['dt'])
forcedcape = series(forcedhistory, 'cape')
forcedrain = series(forcedhistory, 'precip_conv', scale=86400)
controlcape = np.full_like(forcedcape, baselinecape)
controlrain = np.full_like(forcedrain, baselinerain)

fig, axes = plt.subplots(3, 1, figsize=(9, 8), sharex=True)
axes[0].plot(timeaxis, controlcape, color=colors['cloud'], linewidth=2, label='equilibrium baseline')
axes[0].plot(timeaxis, forcedcape, color=colors['heat'], linewidth=3, label='forced column')
axes[0].set_ylabel('CAPE (J kg$^{-1}$)')
axes[0].legend()
axes[1].plot(timeaxis, controlrain, color=colors['cloud'], linewidth=2, label='equilibrium baseline')
axes[1].plot(timeaxis, forcedrain, color=colors['rain'], linewidth=3, label='forced column')
axes[1].set_ylabel('deep rain (mm day$^{-1}$)')
axes[1].legend()
axes[2].plot(timeaxis, series(forcedhistory, 'forcing_energy_tendency'),
             color=colors['sky'], linewidth=3)
axes[2].axhline(0, color=colors['storm'], linewidth=0.8)
axes[2].set(xlabel='model day', ylabel='imposed energy tendency (W m$^{-2}$)')
for ax in axes:
    ax.axvspan(0, forcingdays, color=colors['sky'], alpha=0.10)
    ax.grid(alpha=0.2)
axes[0].text(0.02, 0.88, 'ASCENT ON', transform=axes[0].transAxes,
             color=colors['sky'], fontweight='bold')
fig.suptitle('The column takes a hit, then recovers', fontsize=16, color=colors['storm'])
fig.tight_layout()
plt.show()
capeincrease = forcedcape.max() - baselinecape
rainincrease = forcedrain.max() - baselinerain
capfraction = series(forcedhistory, 'mass_flux_cap_active').mean()
goals = [
    250 <= capeincrease <= 450,
    0.15 <= rainincrease <= 0.30,
    capfraction == 0,
]
ratings = ['0 of 3 constraints met', '1 of 3 met', '2 of 3 met', 'all 3 met']
print(f'maximum ascent: {-peakomega * 36:.2f} hPa hour-1')
print(f"unforced experiment deep rain: {controlrain[0]:.2f} mm day-1")
print(f"forced mean deep rain: {forcedrain.mean():.2f} mm day-1")
print(f'peak CAPE increase: {capeincrease:.1f} J kg-1')
print(f'peak rain increase: {rainincrease:.2f} mm day-1')
print(f'mass-flux limiter active fraction: {capfraction:.0%}')
print(f'design-target rating: {ratings[sum(goals)]}')
print(f'forced-run runtime: {forcedelapsed:.1f} s')


In [ ]:
#@title Play the column response { display-mode: "form" }
# GitHub may suppress JavaScript, so this phase portrait is the static summary.
fig, ax = plt.subplots(figsize=(8, 6))
ax.plot(forcedcape, forcedrain, color=colors['cloud'], linewidth=2, alpha=0.8)
points = ax.scatter(forcedcape, forcedrain, c=timeaxis, cmap='plasma', s=65,
                    edgecolor='white', linewidth=0.6, zorder=3)
ax.scatter(baselinecape, baselinerain, marker='*', s=350, color=colors['sun'],
           edgecolor=colors['storm'], label='equilibrium start', zorder=4)
peakindex = int(np.argmax(forcedcape))
lateindex = int(np.argmin(np.abs(timeaxis - 2.0)))
for index, label, offset in [
    (peakindex, 'peak CAPE', (-72, 12)),
    (lateindex, 'recovery', (10, -18)),
]:
    ax.annotate(label, (forcedcape[index], forcedrain[index]), xytext=offset,
                textcoords='offset points', fontsize=9, color=colors['storm'],
                arrowprops={'arrowstyle': '->', 'color': colors['storm']})
ax.set(xlabel='CAPE (J kg$^{-1}$)', ylabel='deep rain (mm day$^{-1}$)',
       title='Storm phase portrait: does rain chase CAPE?')
ax.grid(alpha=0.2)
ax.legend(frameon=False)
fig.colorbar(points, ax=ax, label='model day')
fig.tight_layout()
plt.show()

frames = np.unique(np.linspace(0, len(timeaxis) - 1, min(30, len(timeaxis))).astype(int))

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
capelines = [
    axes[0].plot([], [], color=colors['cloud'], linewidth=2.5, label='control')[0],
    axes[0].plot([], [], color=colors['heat'], linewidth=3, label='forced')[0],
]
rainlines = [
    axes[1].plot([], [], color=colors['cloud'], linewidth=2.5, label='control')[0],
    axes[1].plot([], [], color=colors['rain'], linewidth=3, label='forced')[0],
]
for ax, values, ylabel, title in [
    (axes[0], [controlcape, forcedcape], 'CAPE (J kg$^{-1}$)', 'Fuel available'),
    (axes[1], [controlrain, forcedrain], 'deep rain (mm day$^{-1}$)', 'Convective response'),
]:
    lower = min(array.min() for array in values)
    upper = max(array.max() for array in values)
    margin = max((upper - lower) * 0.12, 0.02)
    ax.set(xlim=(timeaxis[0], timeaxis[-1]), ylim=(lower - margin, upper + margin),
           xlabel='model day', ylabel=ylabel, title=title)
    ax.axvspan(0, forcingdays, color=colors['sky'], alpha=0.10, label='ascent on')
    ax.grid(alpha=0.2)
    ax.legend(loc='best')
axes[2].set(xlim=(forcedcape.min() - 20, forcedcape.max() + 20),
            ylim=(forcedrain.min() - 0.02, forcedrain.max() + 0.02),
            xlabel='CAPE (J kg$^{-1}$)', ylabel='deep rain (mm day$^{-1}$)',
            title='Storm trajectory')
axes[2].grid(alpha=0.2)
axes[2].scatter(baselinecape, baselinerain, marker='*', s=220,
                color=colors['sun'], edgecolor=colors['storm'])
phaseline = axes[2].plot([], [], color=colors['storm'], linewidth=2.5)[0]
phasepoint = axes[2].plot([], [], 'o', color=colors['heat'], markersize=9)[0]
clock = fig.suptitle('', fontsize=16, color=colors['storm'])

def draw(frame):
    end = frame + 1
    capelines[0].set_data(timeaxis[:end], controlcape[:end])
    capelines[1].set_data(timeaxis[:end], forcedcape[:end])
    rainlines[0].set_data(timeaxis[:end], controlrain[:end])
    rainlines[1].set_data(timeaxis[:end], forcedrain[:end])
    phaseline.set_data(forcedcape[:end], forcedrain[:end])
    phasepoint.set_data([forcedcape[frame]], [forcedrain[frame]])
    status = 'large-scale ascent ON' if timeaxis[frame] <= forcingdays else 'large-scale ascent OFF'
    clock.set_text(f'Day {timeaxis[frame]:.2f}  |  {status}')
    return capelines + rainlines + [phaseline, phasepoint, clock]

movie = animation.FuncAnimation(fig, draw, frames=frames, interval=140, blit=False)
plt.close(fig)
display(HTML(movie.to_jshtml()))

### Design-target exercise

The cell above rates a configuration against three design constraints. Aim to satisfy all three:

- peak CAPE increase between 250 and 450 J kg$^{-1}$;
- peak deep-rain increase between 0.15 and 0.30 mm day$^{-1}$;
- no activation of the mass-flux limiter.

The bounds are a classroom design target, not universal thresholds for real storms.

1. Move only the ascent-strength slider. Describe the sensitivity of CAPE and rain, including whether it appears linear.
2. Return to the default and move only the convective-timescale slider. Why can slower convection increase CAPE while decreasing rainfall?
3. Find a configuration meeting all three constraints that is not the default. Record all three slider values.
4. Find a second, substantially different configuration that also meets them. Compare its phase portrait with the first.
5. Explain the **parameter degeneracy**: how can different parameter combinations satisfy the same constraints through different physical pathways?
6. Deliberately activate or approach a numerical limiter if possible. Explain why that solution should be rejected even if its CAPE and rain meet the target.

Save both configurations and one figure that best demonstrates their different dynamics.


## Mission 6: the storm ingredient laboratory

Convection responds to more than temperature alone. A useful measure of the energy carried by warm, moist air is moist static energy,

$$h=c_pT+gz+L_vq.$$

Near the surface, heating raises the $c_pT$ contribution while moistening raises the $L_vq$ contribution. Both can increase parcel buoyancy and CAPE, but moisture also supplies the condensate needed for precipitation. Because convection has thresholds and feedbacks, the response to heating and moistening together need not equal the sum of their separate responses.

Run a four-member experiment:

1. control;
2. boundary-layer heating only;
3. boundary-layer moistening only;
4. heating and moistening together.

All four columns start from the same sounding and run simultaneously. The imposed tendencies are concentrated below roughly 850 hPa and switched off after the selected duration.

**Predict first:** Rank the four cases by peak CAPE and peak rainfall. Will the combined response be additive, greater than additive, or less than additive?


In [ ]:
heatingrate = 1.0 #@param {type:"slider", min:0.0, max:3.0, step:0.25}
moisteningrate = 1.0 #@param {type:"slider", min:0.0, max:3.0, step:0.25}
ingredientduration = 1.0 #@param {type:"slider", min:0.25, max:2.0, step:0.25}

In [ ]:
#@title Run the four-column ingredient experiment { display-mode: "form" }
ingredientnames = ['control', 'heating', 'moistening', 'both']
heatfactors = torch.tensor([0.0, 1.0, 0.0, 1.0], device=device).reshape(-1, 1)
moisturefactors = torch.tensor([0.0, 0.0, 1.0, 1.0], device=device).reshape(-1, 1)
ingredientgrid = make_grid(experiment['nlevels'], device=device)
ingredientsigma = ingredientgrid['sigma_full'].to(device=device).reshape(1, -1)
ingredientshape = torch.exp(-0.5 * ((ingredientsigma - 0.92) / 0.08) ** 2)
ingredientshape = ingredientshape / ingredientshape.max()

def ingredientforcing(step, state):
    if step * experiment['dt'] >= ingredientduration * 86400:
        return None
    temperaturetendency = heatfactors * ingredientshape * heatingrate / 86400
    moisturetendency = moisturefactors * ingredientshape * moisteningrate / 1000 / 86400
    return {
        'dt': temperaturetendency.to(state['t'].dtype),
        'dq': moisturetendency.to(state['q'].dtype),
    }

ingredientsettings = dict(experiment)
ingredientsettings['days'] = 3
ingredientstart = loadreference(experiment['nlevels'], batch=4)
grid, params, ingredientstate, ingredienthistory, ingredientelapsed = integrate(
    ingredientsettings, batch=4, state=ingredientstart,
    lsforcing=ingredientforcing,
)
ingredienttime = days(ingredienthistory, ingredientsettings['dt'])
ingredientcolors = [colors['cloud'], colors['heat'], colors['rain'], colors['storm']]
controlcape = series(ingredienthistory, 'cape', member=0)
controlrain = series(ingredienthistory, 'precip_conv', member=0, scale=86400)
capepeaks = []
rainresponses = []

fig, axes = plt.subplots(2, 2, figsize=(12, 9))
for member, (name, color) in enumerate(zip(ingredientnames, ingredientcolors)):
    cape = series(ingredienthistory, 'cape', member=member)
    rain = series(ingredienthistory, 'precip_conv', member=member, scale=86400)
    capeanomaly = cape - controlcape
    rainanomaly = rain - controlrain
    capepeaks.append(capeanomaly.max())
    rainresponses.append(rainanomaly.mean())
    axes[0, 0].plot(ingredienttime, capeanomaly, color=color, linewidth=2.7, label=name)
    axes[1, 0].plot(ingredienttime, rainanomaly, color=color, linewidth=2.7, label=name)

matrixorder = [0, 2, 1, 3]
capegrid = np.array(capepeaks)[matrixorder].reshape(2, 2)
raingrid = np.array(rainresponses)[matrixorder].reshape(2, 2)
for ax, values, title, label, colormap in [
    (axes[0, 1], capegrid, 'Peak CAPE response', 'J kg$^{-1}$', 'magma'),
    (axes[1, 1], raingrid, 'Mean rain response', 'mm day$^{-1}$', 'RdBu_r'),
]:
    limit = max(np.abs(values).max(), 1e-9)
    limits = (-limit, limit) if values.min() < 0 else (0, limit)
    image = ax.imshow(values, origin='lower', cmap=colormap, aspect='auto',
                      vmin=limits[0], vmax=limits[1])
    ax.set_xticks([0, 1], ['dry', 'moistened'])
    ax.set_yticks([0, 1], ['unheated', 'heated'])
    for row in range(2):
        for column in range(2):
            ax.text(column, row, f'{values[row, column]:.2f}', ha='center', va='center',
                    color='white' if abs(values[row, column]) > 0.55 * limit else colors['storm'],
                    fontweight='bold')
    ax.set_title(title)
    fig.colorbar(image, ax=ax, label=label, shrink=0.8)

axes[0, 0].set(ylabel='CAPE anomaly (J kg$^{-1}$)', title='Instability through time')
axes[1, 0].set(xlabel='model day', ylabel='deep-rain anomaly (mm day$^{-1}$)',
               title='Rain response through time')
for ax in axes[:, 0]:
    ax.axvspan(0, ingredientduration, color=colors['sun'], alpha=0.08)
    ax.axhline(0, color='black', linewidth=0.8)
    ax.grid(alpha=0.2)
    ax.legend(frameon=False, ncol=2)
fig.suptitle('Storm ingredient laboratory', fontsize=17, color=colors['storm'])
fig.tight_layout()
plt.show()

capenergy = cp * heatingrate
moistureenergy = Lv * moisteningrate / 1000
capesynergy = capepeaks[3] - capepeaks[1] - capepeaks[2]
rainsynergy = rainresponses[3] - rainresponses[1] - rainresponses[2]
ingredientcap = series(ingredienthistory, 'mass_flux_cap_active', member=3).mean()
print(f'heating contribution to h: {capenergy:.0f} J kg-1 day-1 at profile maximum')
print(f'moistening contribution to h: {moistureenergy:.0f} J kg-1 day-1 at profile maximum')
print(f'combined CAPE nonlinearity: {capesynergy:+.1f} J kg-1')
print(f'combined rain nonlinearity: {rainsynergy:+.3f} mm day-1')
print(f'mass-flux limiter active fraction (heated+moistened): {ingredientcap:.0%}')
print(f'batched ingredient runtime: {ingredientelapsed:.1f} s')

### Mission 6 tasks

1. Convert the imposed heating and moistening into their contributions to $h$. Which slider adds more moist static energy at the default settings?
2. Explain separately how low-level heating and low-level moistening alter parcel buoyancy, CAPE, and the water available for rain.
3. Use the printed nonlinearity values to decide whether the combined response is additive, super-additive, or sub-additive. Explain the sign physically.
4. Hold heating fixed and increase moistening. Does CAPE, rainfall, or both respond most strongly? Then reverse the experiment.
5. Approximately 1 g kg$^{-1}$ of moistening adds the same $h$ as 2.5 K of heating. Compare those two cases. Does equal imposed $h$ produce equal CAPE and rain?
6. **Ingredient challenge:** maximize the peak rain response without activating the limiter. Record the three slider values and explain why simply maximizing every slider is not a satisfying scientific strategy.

Include one sentence explaining why rainfall requires both a dynamical trigger and a moisture supply, even when CAPE is already present.

## Mission 7: how long does the atmosphere remember?

The three-day experiment ended while CAPE and rainfall were still displaced from equilibrium. That raises a dynamical question: **does the column remember how long it was forced after the forcing disappears?**

Run three columns for eight days. Give each the same ascent strength but switch ascent off after 0.5, 1, or 2 days. Because the cases run together as a batch, the experiment is much faster than three separate notebook runs.

**Predict first:** Rank the three cases by peak CAPE, peak rain, and recovery time. Is the response controlled only by the instantaneous value of $\omega$, or does the accumulated forcing history matter?

The controls below are intentionally visible. Start with the defaults, then edit them for the storm-design task.


In [ ]:
memorydays = 8 #@param {type:"slider", min:5, max:15, step:1}
memoryomega = 0.05 #@param {type:"slider", min:0.01, max:0.10, step:0.01}
shortpulse = 0.50 #@param {type:"slider", min:0.25, max:1.00, step:0.25}
middlepulse = 1.00 #@param {type:"slider", min:0.50, max:2.00, step:0.25}
longpulse = 2.00 #@param {type:"slider", min:1.00, max:4.00, step:0.50}

memoryomegavalues = [memoryomega, memoryomega, memoryomega]
durationvalues = [shortpulse, middlepulse, longpulse]

In [ ]:
#@title Run the atmospheric memory experiment { display-mode: "form" }
baselinecape, baselinerain = baselinecolumn()
memorysettings = dict(experiment)
memorysettings['days'] = memorydays
memorysettings['diagnostic_hours'] = 6
memorygrid = make_grid(experiment['nlevels'], device=device)
memorystart = loadreference(experiment['nlevels'], batch=len(durationvalues))
memoryforcing = ascentforcing(
    memorygrid, memoryomegavalues, durationdays=durationvalues
)
grid, params, memorystate, memoryhistory, memoryelapsed = integrate(
    memorysettings, batch=len(durationvalues), state=memorystart,
    lsforcing=memoryforcing,
)
memorytime = days(memoryhistory, memorysettings['dt'])
memorycolors = plt.cm.plasma(np.linspace(0.15, 0.85, len(durationvalues)))

fig, axes = plt.subplots(2, 1, figsize=(10, 8), sharex=True)
memoryresults = []
for member, (omega, duration, color) in enumerate(
    zip(memoryomegavalues, durationvalues, memorycolors)
):
    cape = series(memoryhistory, 'cape', member=member)
    rain = series(memoryhistory, 'precip_conv', member=member, scale=86400)
    capeanomaly = cape - baselinecape
    label = f'{duration:g}-day pulse'
    axes[0].plot(memorytime, capeanomaly, color=color, linewidth=2.8, label=label)
    axes[1].plot(memorytime, rain, color=color, linewidth=2.8, label=label)
    axes[0].plot(duration, np.interp(duration, memorytime, capeanomaly),
                 marker='x', color=color, markersize=10, markeredgewidth=2)
    axes[1].plot(duration, np.interp(duration, memorytime, rain),
                 marker='x', color=color, markersize=10, markeredgewidth=2)

    peakindex = int(np.argmax(capeanomaly))
    peaklag = memorytime[peakindex] - duration
    peakvalue = capeanomaly[peakindex]
    after = np.where(memorytime >= duration)[0]
    recovered = after[np.abs(capeanomaly[after]) <= 0.1 * max(abs(peakvalue), 1.0)]
    recoveryday = memorytime[recovered[0]] if len(recovered) else np.nan
    memoryresults.append((duration, omega, peakvalue, peaklag, recoveryday))

axes[0].axhline(0, color=colors['storm'], linewidth=1)
axes[1].axhline(baselinerain, color=colors['cloud'], linewidth=2,
                linestyle='--', label='equilibrium rain')
axes[0].set(ylabel='CAPE anomaly (J kg$^{-1}$)', title='Stored instability')
axes[1].set(xlabel='model day', ylabel='deep rain (mm day$^{-1}$)',
            title='Convective response')
for ax in axes:
    ax.grid(alpha=0.2)
    ax.legend(frameon=False, ncol=2)
fig.suptitle('Atmospheric memory: the forcing stops, the response continues',
             fontsize=16, color=colors['storm'])
fig.tight_layout()
plt.show()

print('duration | omega | peak CAPE anomaly | lag after shutoff | 10% recovery')
for duration, omega, peakvalue, peaklag, recoveryday in memoryresults:
    recoverytext = f'day {recoveryday:.1f}' if np.isfinite(recoveryday) else f'not by day {memorydays}'
    print(f'{duration:4.1f} d   | {omega:4.2f}  | {peakvalue:8.1f} J kg-1   | '
          f'{peaklag:+5.2f} d          | {recoverytext}')
print(f'batched memory runtime: {memoryelapsed:.1f} s')

### Mission 7 tasks

1. Mark the forcing shutoff time for each curve. Does CAPE peak before, at, or after shutoff? Explain the sign of the lag.
2. Which variable retains the clearer memory of pulse duration: CAPE or rainfall? Support the answer with two numbers.
3. The table defines recovery as returning within 10% of the largest CAPE anomaly. Decide whether that is a sensible definition and propose an alternative.
4. Explain physically why the response can persist when $\omega$ has returned to zero. Your explanation should mention both stored instability and convective adjustment.
5. **Fair-fight experiment:** set `memoryomegavalues = [0.10, 0.05, 0.025]`. The products of ascent strength and duration are then equal. Run again. If integrated forcing were the only control, all three responses would match. Do they?
6. **Design your own storm:** increase `memorydays` to 10--15 and create one case that recovers quickly and one that retains a long memory. State the parameters and defend your design before running it.

For your report, keep the most revealing memory figure and write one sentence distinguishing an instantaneous forcing from its time-integrated impulse.


## Ascent strength against convective timescale

Two timescales are now in your hands, and they work against each other:

- **dynamical forcing:** stronger ascent destabilizes and moistens the column faster;
- **convective adjustment:** a shorter mass-flux CAPE-relaxation timescale lets parameterized convection respond faster.

Run nine columns spanning three ascent strengths and three convective timescales. Ascent acts for one day and each case runs for two days. Work out which corner of that grid should build the most CAPE, and which should produce the heaviest deep rain.

**Write your answer down before you run the cell.**

> Largest CAPE buildup: ______ ascent and ______ convective response.
>
> Strongest deep rain: ______ ascent and ______ convective response.

All nine columns run simultaneously. The timescale controls the mass-flux CAPE-relaxation time (`tau_cape`); watch the mass-flux limiter diagnostic for cases where the closure saturates.


In [ ]:
omegavalues = [0.02, 0.05, 0.10]
timescalevalues = [7200.0, 21600.0, 43200.0]

In [ ]:
#@title Run the nine-column sweep { display-mode: "form" }
baselinecape, baselinerain = baselinecolumn()
cases = [(omega, timescale) for omega in omegavalues for timescale in timescalevalues]
updates = {
    'tau_cape': torch.tensor([case[1] for case in cases], device=device),
}
challengesettings = dict(experiment)
challengesettings['days'] = 2
challengegrid = make_grid(experiment['nlevels'], device=device)
challengestart = loadreference(experiment['nlevels'], batch=len(cases))
challengeforcing = ascentforcing(
    challengegrid, [case[0] for case in cases], durationdays=1.0
)
grid, params, challengestate, challengehistory, elapsed = integrate(
    challengesettings, updates=updates, batch=len(cases), state=challengestart,
    lsforcing=challengeforcing,
)
capeanomalies = []
rainrates = []
capfractions = []
referencecape = baselinecape
for member, case in enumerate(cases):
    cape = series(challengehistory, 'cape', member=member)
    capeanomalies.append(cape.max() - referencecape)
    rainrates.append(series(challengehistory, 'precip_conv', member=member, scale=86400).mean())
    capfractions.append(series(challengehistory, 'mass_flux_cap_active', member=member).mean())

capegrid = np.array(capeanomalies).reshape(len(omegavalues), len(timescalevalues))
raingrid = np.array(rainrates).reshape(len(omegavalues), len(timescalevalues))
fig, axes = plt.subplots(1, 2, figsize=(12, 5), constrained_layout=True)
for ax, values, title, label, colormap in [
    (axes[0], capegrid, 'maximum CAPE anomaly', 'CAPE anomaly (J kg-1)', 'coolwarm'),
    (axes[1], raingrid, 'mean deep precipitation', 'rain (mm day-1)', 'viridis'),
]:
    image = ax.imshow(values, origin='lower', aspect='auto', cmap=colormap)
    for row in range(values.shape[0]):
        for column in range(values.shape[1]):
            ax.text(column, row, f'{values[row, column]:.1f}', ha='center', va='center',
                    color='white' if abs(values[row, column]) > 0.55 * np.nanmax(abs(values)) else colors['storm'],
                    fontweight='bold')
    ax.set_xticks(range(len(timescalevalues)), [f'{value / 3600:.0f}' for value in timescalevalues])
    ax.set_yticks(range(len(omegavalues)), [f'{value * 36:.2f}' for value in omegavalues])
    ax.set(xlabel='convective timescale (hours)', ylabel='peak ascent magnitude (hPa hour-1)', title=title)
    fig.colorbar(image, ax=ax, label=label)
peakrow, peakcolumn = np.unravel_index(np.argmax(capegrid), capegrid.shape)
axes[0].scatter(peakcolumn, peakrow, marker='*', s=500, facecolors='none',
                edgecolors=colors['sun'], linewidths=3, label='CAPE champion')
axes[0].legend(loc='upper left', bbox_to_anchor=(0, -0.20), frameon=False)
fig.suptitle('Ascent strength against convective timescale', fontsize=17, color=colors['storm'])
plt.show()

print(f'maximum cap-active fraction: {max(capfractions):.0%}')
strongest = int(np.argmax(capeanomalies))
print('largest CAPE accumulation (omega, timescale):', cases[strongest])
print(f'maximum CAPE anomaly: {capeanomalies[strongest]:.1f} J kg-1')
print(f'batched sweep runtime: {elapsed:.1f} s')



## What to hand in

Submit the completed notebook and a short written report organized around five claims:

1. **Mass and stability:** Where is the column most stable, least stable, and most massive?
2. **Process and radiation diagnosis:** Which physics tendencies maintain the sounding, and where does radiation heat or cool it?
3. **Storm ingredients:** How do heating and moistening separately and jointly affect CAPE and rainfall?
4. **Checking your predictions:** Did your predicted signs, and the corner of the sweep you expected to be strongest, match the model? Explain any miss using the equations, not hindsight.
5. **Model limits:** What important atmospheric behavior cannot occur in a single column without a dynamical core?

Include one figure that you consider the strongest evidence for your interpretation and report the parameter choices used to create it.
